In [1]:
import pandas as pd

In [2]:
import openpyxl
import re

In [3]:
wb = openpyxl.load_workbook('table.xlsx')

In [4]:
ws = wb.active

In [5]:
parsed_data = []

In [6]:
rows = list(ws.iter_rows(values_only=True))

In [7]:
data_rows = rows[26:]

In [8]:
for i in range(0, len(data_rows), 3):
    if i + 2 >= len(data_rows):
        break

    row1 = data_rows[i]
    row3 = data_rows[i + 2]

    qty = int(row1[14]) if row1[14] else 0
    amount = float(row1[30]) if row1[30] else 0
    
    messy_string = row3[0]

    if not messy_string.startswith('CN Code'):
        break
        
    CN_regex = re.search(r'CN Code:\s*(\d+)', messy_string)
    Net_regex = re.search(r'Net weight\s\(g\):([\d\s]+)', messy_string)
    Gross_regex = re.search(r'Gross weight\s\(g\):([\d\s]+)', messy_string)

    code = CN_regex.group(1) if CN_regex else None
    net_str = Net_regex.group(1) if Net_regex else None
    gross_str = Gross_regex.group(1) if Gross_regex else None

    net = float(re.sub(r'\s+', '', net_str))/1000
    gross = float(re.sub(r'\s+', '', gross_str))/1000

    parsed_data.append({
        'CN code': code,
        'qty': qty,
        'net': net,
        'gross': gross,
        'amount': amount,
    })

In [9]:
clean_df = pd.DataFrame(parsed_data)

In [ ]:
clean_df.to_excel('clean_data.xlsx', index=False)

In [11]:
indexed = clean_df.set_index('CN code')

In [12]:
grouped = indexed.groupby(indexed.index)

In [13]:
compact = grouped.sum()

In [14]:
compact.loc['Total'] = compact.sum()

In [15]:
print(compact)

              qty      net    gross      amount
CN code                                        
33041000   6589.0   29.968   87.191   952743.68
33042000   7678.0   43.704  131.776  1716354.00
33043000    300.0    3.645    6.216    61152.12
33049100   1623.0   11.823   60.351   431811.18
33049900   3072.0   76.955  169.983   958768.17
35061000     90.0    0.720    1.602    20484.90
67041900    486.0    2.752    6.152   113822.82
67042000     60.0    1.158    1.158    19655.40
82032000    288.0    8.694    8.694    66440.88
82141000    300.0    2.700    2.700    23118.00
96033090    216.0    8.373    8.373    62612.64
96162000    174.0    2.610    2.610    39604.14
Total     20876.0  193.102  486.806  4466567.93


In [16]:
compact.to_excel('calculated.xlsx')